[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/timursingh33/civ1287_vibe_coded_assignment/blob/main/notebooks/road_defect_classification.ipynb)

# Road Defect Classification — CIV1287H Vibe-Coding Assignment

**Goal.** Use a pre-trained image classifier (transfer learning) to label road-surface defect images. The construction-engineering relevance is automated pavement condition assessment for infrastructure inspection and maintenance planning.

**Workflow.** `Input` (dataset) → `Processing` (pre-trained ResNet18, replace classifier head, fine-tune) → `Output` (confusion matrix, per-class metrics, qualitative grid).

**Dataset.** [Kaggle: patelmihir/road-defects-nonaugmented](https://www.kaggle.com/datasets/patelmihir/road-defects-nonaugmented). 4 classes (Cracks, Patch, Potholes, Surface_Defects), 100 images each.

**Hardware.** Local Linux Mint + NVIDIA RTX 3050 Ti (≈4 GB VRAM). All experiments respect that budget (small batches, AMP, 224×224).

## 0. Setup — imports, seeds, device

Fix the random seeds so the train/val/test split, model init, and dataloader shuffling are reproducible. Verify the GPU is actually picked up — if `cuda` is `False`, training will silently fall back to CPU and the experiment timings in the report would be wrong.

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [3]:
import json
import os
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'Free VRAM (MiB): {torch.cuda.mem_get_info()[0] / 1024**2:.0f}')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'road_defects'
OUT_DIR = PROJECT_ROOT / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
CKPT_DIR = OUT_DIR / 'checkpoints'
for d in (OUT_DIR, FIG_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Data dir    :', DATA_DIR)

ModuleNotFoundError: No module named 'pandas'

## 1. Input — locate dataset, inspect classes and counts

`ImageFolder` expects `DATA_DIR/<class>/<image>` layout. The Kaggle archive ships each class wrapped in a same-named subfolder, so `scripts/download_data.py` flattens it via symlinks under `data/road_defects/`.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

image_root = DATA_DIR
assert image_root.is_dir(), (
    f'Dataset not found at {image_root}. Run scripts/download_data.py first.'
)

classes = sorted([d.name for d in image_root.iterdir() if d.is_dir()])
print('Classes:', classes)

counts = {c: sum(1 for f in (image_root / c).iterdir()
                  if f.suffix.lower() in IMG_EXT) for c in classes}
print('Counts per class:')
for c, n in counts.items():
    print(f'  {c:>20s}: {n}')
print('Total images:', sum(counts.values()))

In [ ]:
# Sample image grid — sanity check that classes look like what we expect.
fig, axes = plt.subplots(len(classes), 3, figsize=(9, 3 * len(classes)))
if len(classes) == 1:
    axes = np.array([axes])
for row, c in enumerate(classes):
    files = [f for f in (image_root / c).iterdir() if f.suffix.lower() in IMG_EXT][:3]
    for col, f in enumerate(files):
        axes[row, col].imshow(Image.open(f).convert('RGB'))
        axes[row, col].set_title(f'{c}', fontsize=10)
        axes[row, col].axis('off')
fig.tight_layout()
fig.savefig(FIG_DIR / 'sample_images.png', dpi=120)
plt.show()

## 2. Processing — transforms, splits, model, training

Random shuffle into 70 / 15 / 15 (the class split is already balanced at 100 each, so this is fine without stratification). Train transforms include light augmentation (flip + small rotation + jitter); val/test use deterministic preprocessing. We use the ImageNet mean/std because the pre-trained model was normalised that way — different normalisation will hurt transfer-learning accuracy.

In [ ]:
IMG_SIZE = 480
# higher resolution results in higher processing time
BATCH_SIZE = 32
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = datasets.ImageFolder(str(image_root), transform=train_tf)
full_eval = datasets.ImageFolder(str(image_root), transform=eval_tf)
assert full_train.classes == full_eval.classes
CLASSES = full_train.classes
NUM_CLASSES = len(CLASSES)

n = len(full_train)
rng = np.random.RandomState(SEED)
idx = rng.permutation(n)
n_train = int(0.70 * n)
n_val = int(0.15 * n)
train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

train_ds = Subset(full_train, train_idx)
val_ds = Subset(full_eval, val_idx)
test_ds = Subset(full_eval, test_idx)
print(f'Train/val/test sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

In [ ]:
def make_loaders(batch_size: int = BATCH_SIZE, num_workers: int = NUM_WORKERS):
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                   num_workers=num_workers, pin_memory=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                   num_workers=num_workers, pin_memory=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                   num_workers=num_workers, pin_memory=True),
    )

train_loader, val_loader, test_loader = make_loaders()
print('Batches:', len(train_loader), len(val_loader), len(test_loader))

In [ ]:
def build_model(arch: str, num_classes: int) -> nn.Module:
    """Return an ImageNet-pretrained backbone with its classifier head swapped.

    Backbone weights start frozen; the new head is trained first.
    """
    if arch == 'resnet18':
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'mobilenet_v3_small':
        m = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
    else:
        raise ValueError(arch)
    return m

def trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(x)
            loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += x.size(0)
        all_preds.append(pred.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    return (
        loss_sum / total,
        correct / total,
        np.concatenate(all_preds),
        np.concatenate(all_labels),
    )

In [ ]:
def run_experiment(arch: str, epochs_head: int = 5, epochs_ft: int = 3,
                    lr_head: float = 1e-3, lr_ft: float = 1e-4,
                    label: str | None = None) -> dict:
    """Train a model: head-only first, then a short unfreeze pass."""
    label = label or arch
    model = build_model(arch, NUM_CLASSES).to(DEVICE)
    print(f'\n=== {label} ===')
    print(f'Trainable params (head only): {trainable_params(model):,}')

    # class-weighted loss to handle imbalance (no-op here since classes are balanced,
    # but it keeps the recipe correct if you swap in an imbalanced dataset)
    label_counter = Counter(full_train.targets[i] for i in train_idx)
    weights = torch.tensor(
        [1.0 / label_counter[i] for i in range(NUM_CLASSES)],
        dtype=torch.float32, device=DEVICE,
    )
    weights = weights / weights.sum() * NUM_CLASSES
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr_head)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

    history = []
    t0 = time.time()
    for epoch in range(epochs_head):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        history.append({'phase': 'head', 'epoch': epoch + 1,
                         'train_loss': tr_loss, 'train_acc': tr_acc,
                         'val_loss': vl_loss, 'val_acc': vl_acc})
        print(f'[head {epoch+1}/{epochs_head}] tr_loss={tr_loss:.3f} tr_acc={tr_acc:.3f} '
               f'val_loss={vl_loss:.3f} val_acc={vl_acc:.3f}')

    # Unfreeze later layers for a brief fine-tune
    if epochs_ft > 0:
        if arch == 'resnet18':
            for p in model.layer4.parameters():
                p.requires_grad = True
        else:
            for p in model.features[-3:].parameters():
                p.requires_grad = True
        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad], lr=lr_ft)
        print(f'Trainable params (fine-tune): {trainable_params(model):,}')
        for epoch in range(epochs_ft):
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
            vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
            history.append({'phase': 'finetune', 'epoch': epoch + 1,
                             'train_loss': tr_loss, 'train_acc': tr_acc,
                             'val_loss': vl_loss, 'val_acc': vl_acc})
            print(f'[ft   {epoch+1}/{epochs_ft}] tr_loss={tr_loss:.3f} tr_acc={tr_acc:.3f} '
                   f'val_loss={vl_loss:.3f} val_acc={vl_acc:.3f}')

    train_seconds = time.time() - t0
    te_loss, te_acc, preds, labels = evaluate(model, test_loader, criterion)
    print(f'TEST acc={te_acc:.3f}  ({train_seconds:.0f}s training)')

    return {
        'label': label,
        'arch': arch,
        'history': history,
        'test_loss': te_loss,
        'test_acc': te_acc,
        'train_seconds': train_seconds,
        'preds': preds,
        'labels': labels,
        'model': model,
    }

### 2a. Primary experiment — ResNet18 with frozen backbone + brief fine-tune

In [ ]:
result_resnet = run_experiment('resnet18', epochs_head=5, epochs_ft=3, label='ResNet18 (head + ft)')

### 2b. Variation — MobileNetV3-Small (smaller, faster backbone for comparison)

Required by the assignment: compare across at least one variation. MobileNetV3-Small is a much smaller backbone — useful for thinking about edge deployment in field-inspection scenarios.

In [ ]:
result_mobile = run_experiment('mobilenet_v3_small', epochs_head=5, epochs_ft=3,
                                label='MobileNetV3-Small (head + ft)')

## 3. Output — confusion matrix, per-class metrics, qualitative grid, metrics.json

Save everything the evaluation report will reference, so the numbers in the report come from real outputs and not from memory.

In [ ]:
def summarise(result: dict) -> dict:
    preds, labels = result['preds'], result['labels']
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        labels, preds, labels=list(range(NUM_CLASSES)),
        target_names=CLASSES, output_dict=True, zero_division=0,
    )
    return {
        'label': result['label'],
        'arch': result['arch'],
        'test_acc': result['test_acc'],
        'test_loss': result['test_loss'],
        'train_seconds': result['train_seconds'],
        'confusion_matrix': cm.tolist(),
        'classification_report': report,
        'history': result['history'],
    }

summary_resnet = summarise(result_resnet)
summary_mobile = summarise(result_mobile)

In [ ]:
def plot_confusion(summary: dict, fname: str) -> None:
    cm = np.array(summary['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(0.7 * NUM_CLASSES + 3, 0.7 * NUM_CLASSES + 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f"{summary['label']} — test acc {summary['test_acc']:.3f}")
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=120)
    plt.show()

plot_confusion(summary_resnet, 'confusion_resnet18.png')
plot_confusion(summary_mobile, 'confusion_mobilenetv3.png')

In [ ]:
# Per-class metric table for the report
def per_class_table(summary: dict) -> pd.DataFrame:
    rep = summary['classification_report']
    rows = []
    for c in CLASSES:
        r = rep[c]
        rows.append({
            'class': c,
            'precision': r['precision'],
            'recall': r['recall'],
            'f1': r['f1-score'],
            'support': int(r['support']),
        })
    return pd.DataFrame(rows)

print('--- ResNet18 ---')
print(per_class_table(summary_resnet).to_string(index=False))
print('\n--- MobileNetV3-Small ---')
print(per_class_table(summary_mobile).to_string(index=False))

In [ ]:
# Qualitative grid: a few correct and a few incorrect test predictions from the best model.
best = result_resnet if result_resnet['test_acc'] >= result_mobile['test_acc'] else result_mobile
preds, labels = best['preds'], best['labels']
correct_idx = np.where(preds == labels)[0]
wrong_idx = np.where(preds != labels)[0]
rng2 = np.random.RandomState(SEED)
show_correct = rng2.choice(correct_idx, size=min(6, len(correct_idx)), replace=False)
show_wrong = rng2.choice(wrong_idx, size=min(6, len(wrong_idx)), replace=False) if len(wrong_idx) else np.array([], dtype=int)

def show_grid(indices, title, fname):
    n = len(indices)
    if n == 0:
        print(f'No samples to show for: {title}')
        return
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.atleast_2d(axes)
    for ax, i in zip(axes.flat, indices):
        path, _ = full_eval.samples[test_idx[i]]
        ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(f'true: {CLASSES[labels[i]]}\npred: {CLASSES[preds[i]]}', fontsize=9)
        ax.axis('off')
    for ax in axes.flat[n:]:
        ax.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=120)
    plt.show()

show_grid(show_correct, f"Correct predictions — {best['label']}", 'qualitative_correct.png')
show_grid(show_wrong, f"Incorrect predictions — {best['label']}", 'qualitative_wrong.png')

In [ ]:
# Side-by-side comparison + persist metrics.json (everything the report needs)
comparison = pd.DataFrame([
    {
        'model': summary_resnet['label'],
        'test_accuracy': summary_resnet['test_acc'],
        'macro_f1': summary_resnet['classification_report']['macro avg']['f1-score'],
        'weighted_f1': summary_resnet['classification_report']['weighted avg']['f1-score'],
        'train_seconds': summary_resnet['train_seconds'],
    },
    {
        'model': summary_mobile['label'],
        'test_accuracy': summary_mobile['test_acc'],
        'macro_f1': summary_mobile['classification_report']['macro avg']['f1-score'],
        'weighted_f1': summary_mobile['classification_report']['weighted avg']['f1-score'],
        'train_seconds': summary_mobile['train_seconds'],
    },
])
print(comparison.to_string(index=False))

metrics_payload = {
    'seed': SEED,
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'classes': CLASSES,
    'class_counts': counts,
    'split': {'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)},
    'experiments': [summary_resnet, summary_mobile],
    'comparison': comparison.to_dict(orient='records'),
}
(OUT_DIR / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2))
print('\nWrote', OUT_DIR / 'metrics.json')

In [ ]:
# Save best checkpoint for reproducibility
torch.save(best['model'].state_dict(), CKPT_DIR / f"best_{best['arch']}.pt")
print('Saved checkpoint:', CKPT_DIR / f"best_{best['arch']}.pt")

## 4. Notes for the evaluation report

- Final numbers come from `outputs/metrics.json`. The evaluation report should quote test accuracy, macro-F1, per-class precision/recall, and call out the largest off-diagonal confusion cell.
- Engineering interpretation: in pavement-condition assessment, **false negatives** (missing a real defect) are usually more costly than false positives, because they delay maintenance and let damage compound. Report the per-class recall with that framing.
- The MobileNetV3 comparison is the variation required by Part A — record whether the smaller model trades meaningful accuracy for the much smaller parameter count.